In [8]:
from datasets import load_dataset
from components.models.sentiment_analysis.Dlsta import Dlsta
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np


In [9]:
def filter_single_label(example):
    if len(example["labels"]) > 0:
        example["labels"] = example["labels"][0]
    else:
        example["labels"] = -1  
    return example


In [10]:
def tokenizer(texts):
    tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
    tokenizer.fit_on_texts(texts)   
    return tokenizer.texts_to_sequences(texts)


In [11]:
def separate_data(data):
    texts = [item["text"] for item in data]
    labels = [item["labels"] for item in data]
    return labels, texts

In [12]:
def encode_label(labels, num_classes=28):
    y = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        y[i, label] = 1
    return y

In [13]:
def train():
    dlsta = Dlsta()
    ds = load_dataset("google-research-datasets/go_emotions", "simplified")
    data = ds["train"]
    
    data = data.map(filter_single_label)  
    
    labels, texts = separate_data(data)
    
    y = encode_label(labels=labels)
    tokens = tokenizer(texts)
    
    x_test,y_test = dlsta.train_model( sequences=tokens, y=y)
    dlsta.test_model(y_test=y_test,x_text=x_test)
    return dlsta

In [14]:
ia = train()

39069
Epoch 1/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 162s 110ms/step - accuracy: 0.2750 - loss: 0.1601
Epoch 2/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 174s 124ms/step - accuracy: 0.4625 - loss: 0.1041
Epoch 3/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 164s 118ms/step - accuracy: 0.5155 - loss: 0.0899
Epoch 4/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 167s 119ms/step - accuracy: 0.5792 - loss: 0.0773
Epoch 5/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 180s 129ms/step - accuracy: 0.6484 - loss: 0.0661
Epoch 6/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 172s 123ms/step - accuracy: 0.6991 - loss: 0.0577
Epoch 7/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 183s 131ms/step - accuracy: 0.7553 - loss: 0.0487
Epoch 8/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 186s 133ms/step - accuracy: 0.7953 - loss: 0.0413
Epoch 9/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 196s 141ms/step - accuracy: 0.8352 - loss: 0.0343
Epoch 10/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 193s 138ms/step - accuracy: 0.8565 - loss: 0.0297
Epoch 11/15
1396/1396 ━━━━━━━━━━━━━━━━━━━━ 187s 134ms/step - accuracy: 0.

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 142, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 142, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 28)             │         7,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,146,838 (31.08 MB)

 Trainable params: 2,715,612 (10.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,431,226 (20.72 MB)

TypeError: Dlsta.test_model() got an unexpected keyword argument 'x_text'

In [8]:
def test(dlsta):
    dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
    labels_names = dataset["train"].features["labels"].feature.names
    test_data = dataset["test"]
    labels, texts = separate_data(test_data)

    tokens= tokenizer(texts)

    dlsta.test_model(y_test=labels,x_text=tokens)
    predict = (dlsta.predict(x_text=tokens))
    feels = np.argmax(predict, axis=1)
    dlsta.accuracy(y_predict=feels,y_test=labels)
    for i in range (0,len(feels)):
        for j in range (0, len(labels[i])):
            print("Sentiment testé  : "+labels_names[labels[i][j]])
        print("Sentiment prédit  : "+labels_names[feels[i]])
        print("texte : "+ texts[i])
    
    return feels

In [9]:
feel = test(ia)

170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step
Sentiment testé  : sadness
Sentiment prédit  : neutral
texte : I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!
Sentiment testé  : admiration
Sentiment prédit  : neutral
texte : It's wonderful because it's awful. At not with.
Sentiment testé  : excitement
Sentiment prédit  : anger
texte : Kings fan here, good luck to you guys! Will be an interesting game to watch! 
Sentiment testé  : gratitude
Sentiment prédit  : neutral
texte : I didn't know that, thank you for teaching me something today!
Sentiment testé  : neutral
Sentiment prédit  : neutral
texte : They got bored from haunting earth for thousands of years and ultimately moved on to the afterlife.
Sentiment testé  : gratitude
Sentiment prédit  : joy
texte : Thank you for asking questions and recognizing that there may be things that you don’t know or understand about police tactics. Seriously. Thank you.
Sentiment testé  : gratitude
Sentimen

43410
